# Hakiki Hire: DistilBERT fine-tune (tier 3)

Companion to Part 25 of `02_job_scam_detection.ipynb`. Runs on a free Colab T4.

**Before running:** set Runtime, Change runtime type, T4 GPU.

Upload the three CSVs written by the notebook's 25.3 export cell:
`data/processed/finetune/{train,val,holdout}.csv`

The splits are fingerprint-grouped and the validation and holdout sets contain
real EMSCAD rows only, so the numbers produced here are directly comparable to
the ladder in Parts 24 and 25.

Returns `metrics.json` and `predictions.csv` to load back into 25.3.


In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q transformers datasets accelerate

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [4]:
from google.colab import files
import pandas as pd, numpy as np, torch, json

uploaded = files.upload()  # select train.csv, val.csv, holdout.csv

train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")
holdout = pd.read_csv("holdout.csv")

for name, df in [("train", train), ("val", val), ("holdout", holdout)]:
    print(f"{name:8} {len(df):6,} rows  positive rate {df.target.mean():.2%}")

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print("\nGPU:", torch.cuda.get_device_name(0))

Saving holdout.csv to holdout.csv
Saving train.csv to train.csv
Saving val.csv to val.csv
train    10,875 rows  positive rate 4.15%
val       2,331 rows  positive rate 4.38%
holdout   2,331 rows  positive rate 4.89%

GPU: Tesla T4


## Setup

`max_length=256` covers most postings while keeping the run inside a Colab
session. `class_weight` has no direct equivalent in the Trainer API, so the
imbalance is handled with a weighted loss, mirroring `class_weight="balanced"`
in the scikit-learn models.


In [5]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(df):
    ds = Dataset.from_pandas(df[["model_text", "target"]].rename(
        columns={"model_text": "text", "target": "labels"}))
    return ds.map(
        lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True, remove_columns=["text"])

train_ds, val_ds, holdout_ds = map(to_dataset, [train, val, holdout])

# Inverse-frequency weights, equivalent to class_weight="balanced".
counts = np.bincount(train.target)
weights = torch.tensor(len(train) / (2.0 * counts), dtype=torch.float).cuda()
print("Class weights:", weights.tolist())

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/10875 [00:00<?, ? examples/s]

Map:   0%|          | 0/2331 [00:00<?, ? examples/s]

Map:   0%|          | 0/2331 [00:00<?, ? examples/s]

Class weights: [0.5216327905654907, 12.056541442871094]


In [8]:
import inspect
from transformers import DataCollatorWithPadding

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=weights)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    proba = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    return {
        "pr_auc": average_precision_score(labels, proba),
        "roc_auc": roc_auc_score(labels, proba),
        "brier": brier_score_loss(labels, proba),
    }


model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

wanted = dict(
    output_dir="./out",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    seed=SEED,
    report_to="none",
)
accepted = set(inspect.signature(TrainingArguments.__init__).parameters)
args = TrainingArguments(**{k: v for k, v in wanted.items() if k in accepted})
print("Dropped:", sorted(set(wanted) - accepted) or "none")

# Tokenisation left sequences at their natural lengths, so batches must be
# padded at collation time. Dynamic padding is also faster than padding
# everything to 256.
trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
    compute_metrics=compute_metrics,
)
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dropped: ['warmup_ratio']


Epoch,Training Loss,Validation Loss,Pr Auc,Roc Auc,Brier
1,0.645993,0.503789,0.601200,0.916755,0.029835
2,0.397864,0.552877,0.686609,0.938025,0.027440
3,0.234174,0.606011,0.672554,0.938128,0.029768


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1020, training_loss=0.3978522880404603, metrics={'train_runtime': 296.4614, 'train_samples_per_second': 110.048, 'train_steps_per_second': 3.441, 'total_flos': 2160874440576000.0, 'train_loss': 0.3978522880404603, 'epoch': 3.0})

## Validation results

Compared against the linear models from Part 24, all on the same validation set.


In [9]:
val_out = trainer.predict(val_ds)
val_proba = torch.softmax(torch.tensor(val_out.predictions), dim=1)[:, 1].numpy()

metrics = {
    "model": "distilbert-base-uncased",
    "epochs": args.num_train_epochs,
    "max_length": MAX_LENGTH,
    "pr_auc": float(average_precision_score(val.target, val_proba)),
    "roc_auc": float(roc_auc_score(val.target, val_proba)),
    "brier": float(brier_score_loss(val.target, val_proba)),
}

print(f"DistilBERT     PR-AUC {metrics['pr_auc']:.4f}  "
      f"ROC-AUC {metrics['roc_auc']:.4f}  Brier {metrics['brier']:.4f}")
print(f"TF-IDF LR      PR-AUC 0.7987  ROC-AUC 0.9558  Brier 0.0168")
print(f"Linear SVM     PR-AUC 0.7909  ROC-AUC 0.9479  Brier 0.0166")
print(f"\nDifference vs TF-IDF LR: {metrics['pr_auc'] - 0.7987:+.4f} PR-AUC")

DistilBERT     PR-AUC 0.6866  ROC-AUC 0.9380  Brier 0.0274
TF-IDF LR      PR-AUC 0.7987  ROC-AUC 0.9558  Brier 0.0168
Linear SVM     PR-AUC 0.7909  ROC-AUC 0.9479  Brier 0.0166

Difference vs TF-IDF LR: -0.1121 PR-AUC


## The Kenyan probes

The decisive test. Parts 23 and 24 showed the TF-IDF model scores an explicit
M-Pesa registration-fee demand at 0.236, below the threshold at which the
system would flag it, because EMSCAD contains no such postings.

If DistilBERT's pretraining carries knowledge of fee-advance fraud, it should
score these higher without having seen one in training. If it scores them like
the TF-IDF model, that confirms the gap is missing data rather than model
capacity, and no amount of transfer learning substitutes for a labelled local
corpus.


In [10]:
KENYAN_SCAM_PROBES = [
    "Urgent hiring. Pay a refundable registration fee of KSh 2000 via M-Pesa to secure your slot.",
    "Overseas placement to Dubai. Send your passport and processing fee today. No interview required.",
    "Work from home data entry. Send 1500 to till number 8899 for the starter pack.",
]
LEGITIMATE_PROBES = [
    "Software Engineer, Safaricom PLC, Nairobi. BSc Computer Science, three years experience with distributed systems. Apply via the careers portal.",
    "Registered Nurse, Aga Khan University Hospital. Diploma in Nursing, valid NCK licence, two years ward experience.",
    "Finance Officer, Kenya Red Cross. CPA Part II, three years in donor-funded project accounting.",
]
probes = KENYAN_SCAM_PROBES + LEGITIMATE_PROBES

enc = tokenizer(probes, truncation=True, max_length=MAX_LENGTH,
                padding=True, return_tensors="pt").to("cuda")
model.eval()
with torch.no_grad():
    probe_proba = torch.softmax(model(**enc).logits, dim=1)[:, 1].cpu().numpy()

metrics["probe_scores"] = [float(p) for p in probe_proba]

print(pd.DataFrame({
    "posting": [p[:48] + "..." for p in probes],
    "actual": ["scam"] * 3 + ["legitimate"] * 3,
    "distilbert": probe_proba.round(3),
    "tfidf_lr": [0.236, 0.280, 0.982, 0.139, 0.101, 0.154],
    "tfidf_blend": [0.586, 0.555, 0.990, 0.139, 0.101, 0.154],
}).to_string(index=False))

sep = probe_proba[:3].mean() - probe_proba[3:].mean()
print(f"\nDistilBERT separation: {sep:+.3f}")
print(f"TF-IDF LR separation:  {0.499 - 0.131:+.3f}")

                                            posting     actual  distilbert  tfidf_lr  tfidf_blend
Urgent hiring. Pay a refundable registration fee...       scam       0.966     0.236        0.586
Overseas placement to Dubai. Send your passport ...       scam       0.718     0.280        0.555
Work from home data entry. Send 1500 to till num...       scam       0.993     0.982        0.990
Software Engineer, Safaricom PLC, Nairobi. BSc C... legitimate       0.041     0.139        0.139
Registered Nurse, Aga Khan University Hospital. ... legitimate       0.081     0.101        0.101
Finance Officer, Kenya Red Cross. CPA Part II, t... legitimate       0.104     0.154        0.154

DistilBERT separation: +0.817
TF-IDF LR separation:  +0.368


## Export

`metrics.json` and `predictions.csv` go back into
`data/processed/finetune/`, then re-run cell 25.3 in the main notebook.

The holdout is **not** scored here. It stays sealed until Part 26 selects a
single model, so that it is used exactly once.


In [11]:
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame({
    "fingerprint": val.fingerprint,
    "target": val.target,
    "proba": val_proba,
}).to_csv("predictions.csv", index=False)

print(json.dumps(metrics, indent=2))
files.download("metrics.json")
files.download("predictions.csv")

{
  "model": "distilbert-base-uncased",
  "epochs": 3,
  "max_length": 256,
  "pr_auc": 0.6866094463299662,
  "roc_auc": 0.9380250530001144,
  "brier": 0.027440339595098503,
  "probe_scores": [
    0.9655764698982239,
    0.718051016330719,
    0.992864727973938,
    0.041037436574697495,
    0.08064661920070648,
    0.10393037647008896
  ]
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>